In [1]:
import itertools
import random 
import math
import scipy.stats as ss
import numpy as np
import dendropy
from dendropy.calculate import treemeasure


In [47]:
import skbio
from skbio import DistanceMatrix
from skbio.tree import nj
from skbio import TreeNode
from io import StringIO
from dendropy.calculate import treecompare

In [42]:
import skbio.diversity.beta

In [16]:
def p_delta_change(n_states, l, change: bool):
    if not change:
        p_out = 1 / n_states + (n_states - 1) / n_states * math.exp(- (n_states - 1) * l)
    else:
        p_out = 1 / n_states - math.exp(- (n_states - 1) * l) / n_states
    return p_out

def simulate_cn_seq(prev_cn, n_states, l, alpha=1.):
    node_cn = np.empty_like(prev_cn)
    pdd = p_delta_change(n_states, alpha * l, change=False)
    u = random.random()
    if u < pdd:
        node_cn[0] = prev_cn[0]
    else:
        node_cn[0] = random.choice([j for j in range(n_states) if j != prev_cn[0]])

    for m in range(1, len(prev_cn)):
        u = random.random()
        no_change_cn = prev_cn[m] - prev_cn[m-1] + node_cn[m-1]
        if 0 <= no_change_cn < n_states:
            if u < pdd:
                node_cn[m] = no_change_cn
            else:
                node_cn[m] = random.choice([j for j in range(n_states) if j != no_change_cn])
        else:
            node_cn[m] = random.choice([j for j in range(n_states)])
    return node_cn

def simulate_cn(tree, n_sites, n_states, alpha=1.):
    cn = np.empty((len(tree.nodes()), n_sites))
    cn[0, :] = 2
    assert tree.seed_node.label is not None
    for n in tree.preorder_node_iter():
        if n.label != 0:
            cn[n.label] = simulate_cn_seq(cn[n.parent_node.label], n_states, n.edge_length, alpha=alpha)
    return cn

def emit_normalized_obs(cn_seq, mu=1.0, scale=1.0):
    eps = ss.norm(loc=0., scale=scale).rvs(size=len(cn_seq))
    return np.clip(cn_seq / mu + eps, a_min=0., a_max=None)

def emit_raw_obs(cn_seq, lam=100.):
    return ss.poisson.rvs(mu=np.clip(cn_seq, a_min=.01, a_max=None) * lam, size=len(cn_seq))

def label_tree(tree):
    for i, n in enumerate(tree.preorder_node_iter()):
        n.label = i

def rand_dataset(n_cells: int, n_states: int, n_sites: int, alpha=0.02, obs_type='norm') -> dict:
    tree = dendropy.treesim.treesim.birth_death_tree(birth_rate=0.9, death_rate=0.4, num_extant_tips=n_cells)
    label_tree(tree)
    cn = simulate_cn(tree, n_sites, n_states, alpha=alpha)
    obs = np.empty((n_sites, n_cells))
    tax_id_map = {}
    for i, t in enumerate(tree.leaf_node_iter()):
        tax_id_map[t.taxon] = i
        if obs_type == 'pois':
            obs[:, i] = emit_raw_obs(cn[t.label])
        elif obs_type == 'norm':
            obs[:, i] = emit_normalized_obs(cn[t.label], scale=.7)
    data = {
        'obs': obs,
        'tree': tree,
        'cn': cn,
        'tax_id_map': tax_id_map
    }
    return data

def get_distance_matrix(tree):
    taxa = list(tree.taxon_namespace)
    pdc = tree.phylogenetic_distance_matrix()
    n = len(taxa)
    dist_matrix = np.zeros((n, n))
    for i, t1 in enumerate(taxa):
        for j, t2 in enumerate(taxa):
            if i != j:
                dist_matrix[i, j] = pdc(t1, t2)
    return dist_matrix

In [17]:
# Generate dataset
# 20, 8, 10
data = rand_dataset(10, 4, 5)
tree = data['tree']


In [18]:
# Print initial tree in Newick format
initial_tree_newick = tree.as_string(schema="newick")
print("Initial Tree in Newick format:")
print(initial_tree_newick)

Initial Tree in Newick format:
[&R] ((T7:1.9419555240036812,(T1:1.3341813752912328,(T9:0.39870216203407194,T4:0.39870216203407194)5:0.9354792132571608)3:0.6077741487124484)1:1.528619409852706,((T2:0.6998698337865717,((T3:0,T8:0)12:0.0805974665121463,T6:0.0805974665121463)11:0.6192723672744254)9:0.6037134206303989,(T10:0.07520804183283966,T5:0.07520804183283966)16:1.228375212584131)8:2.166991679439416):1.6486391087743089;



In [6]:
# Calculate the initial distance matrix
initial_dist_matrix = get_distance_matrix(tree)

In [7]:
# Create a DistanceMatrix for scikit-bio
ids_new = [leaf.taxon.label for leaf in tree.leaf_node_iter()]
dm_skbio = DistanceMatrix(initial_dist_matrix, ids_new)

In [8]:
# Perform NJ algorithm using scikit-bio
nj_tree = nj(dm_skbio)

In [26]:
# Get NJ tree in Newick format
nj_tree_newick = str(nj_tree)
# print("NJ Tree in Newick format:")
# print(nj_tree_newick)


NJ Tree in Newick format:
(T6:0.086798,(((T7:0.342672,(T1:0.0,T5:0.0):0.342672):1.655612,(T3:1.110295,(T8:0.29258,T10:0.29258):0.817715):0.887988):2.61754,(T2:0.065863,T9:0.065863):0.413952):0.393017,T4:0.086798);



In [61]:
# Convert NJ tree Newick string to scikit-bio TreeNode
nj_tree_skbio = skbio.TreeNode.read([nj_tree_newick])

In [56]:
taxon_namespace = dendropy.TaxonNamespace()

In [27]:
# Convert initial tree Newick string to scikit-bio TreeNode
initial_tree_skbio = skbio.TreeNode.read([initial_tree_newick])

In [57]:
initial_tree_dendropy = dendropy.Tree.get(data=initial_tree_newick, schema="newick", taxon_namespace=taxon_namespace)


In [58]:
nj_tree_dendropy = dendropy.Tree.get(data=nj_tree_newick, schema="newick", taxon_namespace=taxon_namespace)


In [60]:
rf_distance = treecompare.symmetric_difference(initial_tree_dendropy, nj_tree_dendropy)
print("RF Distance:", rf_distance)


RF Distance: 19
